In [1]:
import os
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"
os.environ["_JAVA_OPTIONS"] = "--add-opens=java.base/javax.security.auth=ALL-UNNAMED"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

# Verify Java is found
import subprocess
result = subprocess.run(["java", "-version"], capture_output=True, text=True)
print(result.stderr)  # java -version prints to stderr, this is normal

# Verify both files exist
import os.path
print("winutils.exe found:", os.path.isfile(r"C:\hadoop\bin\winutils.exe"))
print("hadoop.dll found:  ", os.path.isfile(r"C:\hadoop\bin\hadoop.dll"))

Picked up _JAVA_OPTIONS: --add-opens=java.base/javax.security.auth=ALL-UNNAMED
openjdk version "17.0.19" 2026-04-21
OpenJDK Runtime Environment Temurin-17.0.19+10 (build 17.0.19+10)
OpenJDK 64-Bit Server VM Temurin-17.0.19+10 (build 17.0.19+10, mixed mode, sharing)

winutils.exe found: True
hadoop.dll found:   True


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, StringType, TimestampType
 
spark = (
    SparkSession.builder
    .appName("nyc-yellow-taxi-2025-all-months")
    .master("local[*]")
    .config("spark.driver.memory", "4g")          # raise to "6g" if you have 16GB RAM
    .config("spark.sql.shuffle.partitions", "16")  # slightly more for 12 files
    .config("spark.sql.session.timeZone", "America/New_York")
    .config("spark.sql.parquet.datetimeRebaseModeInRead", "CORRECTED")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Session ready")

Spark version: 3.5.1
Session ready


In [3]:
import os, glob
 
DATA_DIR   = "NYC_Data/rawData"
OUTPUT_DIR = "nyc_taxi_cleaned"
 
os.makedirs(OUTPUT_DIR, exist_ok=True)
files = sorted(glob.glob(f"{DATA_DIR}/yellow_tripdata_2025-*.parquet"))
print(f"Found {len(files)} parquet file(s):")
for f in files:
    size_mb = os.path.getsize(f) / 1_048_576
    print(f"  • {os.path.basename(f):45s}  {size_mb:.0f} MB")
 
df_raw = (
    spark.read
    .option("mergeSchema", "true")   
    .parquet(f"{DATA_DIR}/yellow_tripdata_2025-*.parquet")
)
 
df_raw = df_raw.withColumn(
    "source_month",
    F.regexp_extract(F.input_file_name(), r"(\d{4}-\d{2})", 1)
)
 
print()
print("=" * 55)
print("RAW DATA — All months loaded")
print("=" * 55)
total_raw = df_raw.count()
print(f"Total rows (all months, raw): {total_raw:,}")
print(f"Total columns               : {len(df_raw.columns)}")
 
print()
print("Rows per month (raw):")
df_raw.groupBy("source_month").count().orderBy("source_month").show()

Found 12 parquet file(s):
  • yellow_tripdata_2025-01.parquet                56 MB
  • yellow_tripdata_2025-02.parquet                58 MB
  • yellow_tripdata_2025-03.parquet                67 MB
  • yellow_tripdata_2025-04.parquet                64 MB
  • yellow_tripdata_2025-05.parquet                74 MB
  • yellow_tripdata_2025-06.parquet                70 MB
  • yellow_tripdata_2025-07.parquet                64 MB
  • yellow_tripdata_2025-08.parquet                59 MB
  • yellow_tripdata_2025-09.parquet                69 MB
  • yellow_tripdata_2025-10.parquet                72 MB
  • yellow_tripdata_2025-11.parquet                68 MB
  • yellow_tripdata_2025-12.parquet                70 MB

RAW DATA — All months loaded
Total rows (all months, raw): 48,722,602
Total columns               : 21

Rows per month (raw):
+------------+-------+
|source_month|  count|
+------------+-------+
|     2025-01|3475226|
|     2025-02|3577543|
|     2025-03|4145257|
|     2025-04|3970553|
| 

In [4]:
print("=" * 55)
print("· Missing values across ALL months")
print("=" * 55)
 
null_df = df_raw.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_raw.columns]
)
null_row = null_df.collect()[0].asDict()
print(f"{'Column':<30} {'Nulls':>12}  {'%':>6}")
print("-" * 54)
for col, n in null_row.items():
    pct = n / total_raw * 100
    flag = " ⚠️" if pct > 5 else ""
    print(f"{col:<30} {n:>12,}  {pct:>5.1f}%{flag}")
 
print()
print("Key stats — raw data:")
df_raw.select("trip_distance","fare_amount","total_amount","passenger_count").summary(
    "count","min","max","mean"
).show()

· Missing values across ALL months
Column                                Nulls       %
------------------------------------------------------
VendorID                                  0    0.0%
tpep_pickup_datetime                      0    0.0%
tpep_dropoff_datetime                     0    0.0%
passenger_count                  11,611,894   23.8% ⚠️
trip_distance                             0    0.0%
RatecodeID                       11,611,894   23.8% ⚠️
store_and_fwd_flag               11,611,894   23.8% ⚠️
PULocationID                              0    0.0%
DOLocationID                              0    0.0%
payment_type                              0    0.0%
fare_amount                               0    0.0%
extra                                     0    0.0%
mta_tax                                   0    0.0%
tip_amount                                0    0.0%
tolls_amount                              0    0.0%
improvement_surcharge                     0    0.0%
total_amount     

In [5]:
print("=" * 55)
print("· Fixing data types (all months)")
print("=" * 55)
 
df = (
    df_raw
    .withColumn("VendorID",F.col("VendorID").cast(IntegerType()))
    .withColumn("RatecodeID",F.col("RatecodeID").cast(IntegerType()))
    .withColumn("PULocationID",F.col("PULocationID").cast(IntegerType()))
    .withColumn("DOLocationID",F.col("DOLocationID").cast(IntegerType()))
    .withColumn("payment_type",F.col("payment_type").cast(IntegerType()))
    .withColumn("passenger_count",F.col("passenger_count").cast(IntegerType()))
    .withColumn("tpep_pickup_datetime",F.col("tpep_pickup_datetime").cast(TimestampType()))
    .withColumn("tpep_dropoff_datetime", F.col("tpep_dropoff_datetime").cast(TimestampType()))
    .withColumn("trip_distance",F.col("trip_distance").cast(DoubleType()))
    .withColumn("fare_amount",F.col("fare_amount").cast(DoubleType()))
    .withColumn("extra",F.col("extra").cast(DoubleType()))
    .withColumn("mta_tax",F.col("mta_tax").cast(DoubleType()))
    .withColumn("tip_amount",F.col("tip_amount").cast(DoubleType()))
    .withColumn("tolls_amount",F.col("tolls_amount").cast(DoubleType()))
    .withColumn("improvement_surcharge", F.col("improvement_surcharge").cast(DoubleType()))
    .withColumn("total_amount",F.col("total_amount").cast(DoubleType()))
    .withColumn("congestion_surcharge", F.col("congestion_surcharge").cast(DoubleType()))
    .withColumn("Airport_fee",F.col("Airport_fee").cast(DoubleType()))
    .withColumn("cbd_congestion_fee",F.col("cbd_congestion_fee").cast(DoubleType()))
    .withColumn("store_and_fwd_flag", F.col("store_and_fwd_flag").cast(StringType()))
)
print("✓ All columns cast")

· Fixing data types (all months)
✓ All columns cast


In [6]:
print("=" * 55)
print("· Handling missing values (all months)")
print("=" * 55)
 
# Drop rows with unfixable nulls
before = df.count()
df = df.dropna(subset=[
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID"
])
print(f"  Dropped rows (null timestamps / location): {before - df.count():,}")
 

df = df.fillna({
    "congestion_surcharge" : 0.0,
    "Airport_fee"          : 0.0,
    "extra"                : 0.0,
    "mta_tax"              : 0.0,
    "tip_amount"           : 0.0,
    "tolls_amount"         : 0.0,
    "improvement_surcharge": 0.0,
    "passenger_count"      : 1,
    "VendorID"             : -1,
    "RatecodeID"           : -1,
    "store_and_fwd_flag"   : "N",
})
print("  ✓ Filled all surcharge / count / flag nulls")
 
# Recode payment_type 0 → -1
df = df.withColumn(
    "payment_type",
    F.when(F.col("payment_type") == 0, -1).otherwise(F.col("payment_type"))
)
print("  ✓ Recoded payment_type 0 → -1")

· Handling missing values (all months)
  Dropped rows (null timestamps / location): 0
  ✓ Filled all surcharge / count / flag nulls
  ✓ Recoded payment_type 0 → -1


In [7]:
print("=" * 55)
print("· Removing duplicates (all months)")
print("=" * 55)
 
KEY_COLS = [
    "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "PULocationID", "DOLocationID", "fare_amount", "total_amount"
]
 
before = df.count()
df = df.dropDuplicates(KEY_COLS)
print(f"  Duplicates removed: {before - df.count():,}")
print(f"  Rows remaining    : {df.count():,}")

· Removing duplicates (all months)
  Duplicates removed: 10
  Rows remaining    : 48,722,592


In [8]:
print("=" * 55)
print("· Validating & filtering (all months)")
print("=" * 55)
 
def filter_log(df, condition, label):
    before = df.count()
    df = df.filter(condition)
    removed = before - df.count()
    print(f"  {label:<55} removed: {removed:>8,}")
    return df
 
# Compute duration first
df = df.withColumn(
    "trip_duration_minutes",
    ((F.unix_timestamp("tpep_dropoff_datetime") -
      F.unix_timestamp("tpep_pickup_datetime")) / 60).cast(DoubleType())
)
 
# Timestamps
df = filter_log(df,
    F.col("tpep_pickup_datetime") < F.col("tpep_dropoff_datetime"),
    "Pickup before drop-off")
 
# For multi-month: pickup must fall within 2025 (any month)
df = filter_log(df,
    (F.col("tpep_pickup_datetime") >= "2025-01-01") &
    (F.col("tpep_pickup_datetime") <  "2026-01-01"),
    "Pickup date within 2025")
 
df = filter_log(df,
    F.date_format("tpep_pickup_datetime", "yyyy-MM") == F.col("source_month"),
    "Pickup month matches source file month")
 
# Duration
df = filter_log(df,
    (F.col("trip_duration_minutes") >= 1) &
    (F.col("trip_duration_minutes") <= 1440),
    "Trip duration: 1 min – 24 hours")
 
# Distance
df = filter_log(df,
    (F.col("trip_distance") > 0) &
    (F.col("trip_distance") <= 500),
    "Trip distance: > 0 and ≤ 500 miles")
 
# Money
df = filter_log(df,
    (F.col("fare_amount") >= 0) & (F.col("fare_amount") <= 50_000),
    "Fare amount: $0 – $50,000")
 
df = filter_log(df, F.col("total_amount") >= 0, "Total amount ≥ $0")
df = filter_log(df, F.col("Airport_fee") >= 0,  "Airport fee ≥ $0")
df = filter_log(df, F.col("cbd_congestion_fee") >= 0, "CBD congestion fee ≥ $0")
 
# Passengers
df = filter_log(df,
    (F.col("passenger_count") >= 1) & (F.col("passenger_count") <= 9),
    "Passenger count: 1 – 9")
 
# Recode
df = df.withColumn(
    "RatecodeID",
    F.when(F.col("RatecodeID") == 99, -1).otherwise(F.col("RatecodeID"))
)
print(f"  {'Recoded RatecodeID 99 → -1':<55} (no rows dropped)")
 

· Validating & filtering (all months)
  Pickup before drop-off                                  removed:  546,303
  Pickup date within 2025                                 removed:       29
  Pickup month matches source file month                  removed:      185
  Trip duration: 1 min – 24 hours                         removed:  545,443
  Trip distance: > 0 and ≤ 500 miles                      removed: 1,021,665
  Fare amount: $0 – $50,000                               removed: 2,567,289
  Total amount ≥ $0                                       removed:    2,425
  Airport fee ≥ $0                                        removed:        0
  CBD congestion fee ≥ $0                                 removed:       21
  Passenger count: 1 – 9                                  removed:  248,801
  Recoded RatecodeID 99 → -1                              (no rows dropped)


In [9]:
print("=" * 55)
print("· Adding derived columns (all months)")
print("=" * 55)
 
df = (df
    .withColumn("pickup_date",      F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_year",      F.year("tpep_pickup_datetime"))
    .withColumn("pickup_month",     F.month("tpep_pickup_datetime"))
    .withColumn("pickup_day",       F.dayofmonth("tpep_pickup_datetime"))
    .withColumn("pickup_hour",      F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_dayofweek", F.dayofweek("tpep_pickup_datetime"))
    .withColumn("pickup_dayname",   F.date_format("tpep_pickup_datetime", "EEEE"))
    .withColumn("is_weekend",
        F.when(F.dayofweek("tpep_pickup_datetime").isin([1, 7]), True).otherwise(False))
    .withColumn("time_of_day",
        F.when(F.col("pickup_hour").between(5,  11), "Morning")
         .when(F.col("pickup_hour").between(12, 16), "Afternoon")
         .when(F.col("pickup_hour").between(17, 20), "Evening")
         .when(F.col("pickup_hour").between(21, 23), "Night")
         .otherwise("Late Night"))
    .withColumn("avg_speed_mph",
        F.when(F.col("trip_duration_minutes") > 0,
            F.round(F.col("trip_distance") / (F.col("trip_duration_minutes") / 60), 2)
        ).otherwise(None))
    .withColumn("fare_per_mile",
        F.when(F.col("trip_distance") > 0,
            F.round(F.col("fare_amount") / F.col("trip_distance"), 2)
        ).otherwise(None))
    .withColumn("tip_pct",
        F.when(F.col("fare_amount") > 0,
            F.round(F.col("tip_amount") / F.col("fare_amount") * 100, 2)
        ).otherwise(0.0))
    .withColumn("payment_label",
        F.when(F.col("payment_type") == 1,  "Credit Card")
         .when(F.col("payment_type") == 2,  "Cash")
         .when(F.col("payment_type") == 3,  "No Charge")
         .when(F.col("payment_type") == 4,  "Dispute")
         .when(F.col("payment_type") == 5,  "Unknown")
         .otherwise("Unknown"))
    .withColumn("ratecode_label",
        F.when(F.col("RatecodeID") == 1,  "Standard")
         .when(F.col("RatecodeID") == 2,  "JFK")
         .when(F.col("RatecodeID") == 3,  "Newark")
         .when(F.col("RatecodeID") == 4,  "Nassau/Westchester")
         .when(F.col("RatecodeID") == 5,  "Negotiated")
         .when(F.col("RatecodeID") == 6,  "Group Ride")
         .otherwise("Unknown"))
)
 
print("  ✓ All derived columns added")

· Adding derived columns (all months)
  ✓ All derived columns added


In [10]:
print()
print("=" * 60)
print("· Final Quality Report — All 12 months")
print("=" * 60)
 
cleaned_count = df.count()
print(f"  Raw rows (all months)     : {total_raw:>12,}")
print(f"  Cleaned rows (all months) : {cleaned_count:>12,}")
print(f"  Rows removed              : {total_raw - cleaned_count:>12,}  ({(total_raw-cleaned_count)/total_raw*100:.1f}%)")
print(f"  Data retained             : {cleaned_count/total_raw*100:.1f}%")
 
print()
print("  ── Rows per month (cleaned) ──")
df.groupBy("pickup_month", "source_month").count().orderBy("pickup_month").show()
 
print()
print("  ── Key stats per month ──")
df.groupBy("pickup_month").agg(
    F.count("*").alias("trips"),
    F.round(F.avg("trip_distance"), 2).alias("avg_miles"),
    F.round(F.avg("trip_duration_minutes"), 1).alias("avg_duration_min"),
    F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
    F.round(F.avg("tip_pct"), 1).alias("avg_tip_pct"),
    F.round(F.avg("avg_speed_mph"), 1).alias("avg_speed_mph")
).orderBy("pickup_month").show()
 
print()
print("  ── Payment type distribution ──")
df.groupBy("payment_label").count().orderBy("count", ascending=False).show()
 
print()
print("  ── Time of day distribution ──")
df.groupBy("time_of_day").count().orderBy("count", ascending=False).show()
 
print()
print("  ── Weekend vs Weekday trips ──")
df.groupBy("is_weekend").count().show()
 
print()
print("  ── Remaining null check ──")
null_check = df.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]
).collect()[0].asDict()
remaining = {k: v for k, v in null_check.items() if v > 0}
if remaining:
    for col, n in remaining.items():
        print(f"    {col}: {n:,}")
else:
    print("    ✓ Zero nulls remaining across all columns")
 


· Final Quality Report — All 12 months
  Raw rows (all months)     :   48,722,602
  Cleaned rows (all months) :   43,790,431
  Rows removed              :    4,932,171  (10.1%)
  Data retained             : 89.9%

  ── Rows per month (cleaned) ──
+------------+------------+-------+
|pickup_month|source_month|  count|
+------------+------------+-------+
|           1|     2025-01|3219962|
|           2|     2025-02|3276883|
|           3|     2025-03|3794938|
|           4|     2025-04|3639130|
|           5|     2025-05|4057161|
|           6|     2025-06|3835784|
|           7|     2025-07|3464321|
|           8|     2025-08|3153365|
|           9|     2025-09|3804765|
|          10|     2025-10|3910494|
|          11|     2025-11|3613716|
|          12|     2025-12|4019912|
+------------+------------+-------+


  ── Key stats per month ──
+------------+-------+---------+----------------+--------+-----------+-------------+
|pickup_month|  trips|avg_miles|avg_duration_min|avg_fare|avg

In [11]:
print()
print("=" * 55)
print("· Exporting cleaned data (all months)")
print("=" * 55)
 
# Drop the helper column — consumers don't need it
df_export = df.drop("source_month")
 
print("  Writing partitioned parquet...")
(df_export
    .repartition("pickup_month")
    .sortWithinPartitions("tpep_pickup_datetime")
    .write
    .mode("overwrite")
    .partitionBy("pickup_month")
    .parquet(f"{OUTPUT_DIR}/all_months_cleaned")
)
print(f"  ✓ Parquet → {OUTPUT_DIR}/all_months_cleaned/")
print(f"      Structure: pickup_month=1/, pickup_month=2/ …")
 

print()
print("  Writing monthly summary CSV...")
monthly_summary = (
    df.groupBy("pickup_month", "source_month").agg(
        F.count("*").alias("total_trips"),
        F.round(F.sum("fare_amount"), 2).alias("total_fare"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("trip_distance"), 3).alias("avg_distance_miles"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_min"),
        F.round(F.avg("fare_amount"), 3).alias("avg_fare"),
        F.round(F.avg("tip_pct"), 2).alias("avg_tip_pct"),
        F.round(F.avg("avg_speed_mph"), 2).alias("avg_speed_mph"),
    ).orderBy("pickup_month")
)
(monthly_summary.coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(f"{OUTPUT_DIR}/monthly_summary")
)
print(f"  ✓ Monthly summary CSV → {OUTPUT_DIR}/monthly_summary/")
 
print()
print("=" * 55)
print("Phase 2 complete! All 12 months are clean.")
print()
print("  Your output folder:")
print(f"    {OUTPUT_DIR}/")
print(f"    ├── all_months_cleaned/")
print(f"    │   ├── pickup_month=1/")
print(f"    │   ├── pickup_month=2/")
print(f"    │   └── ... (one folder per month)")
print(f"    └── monthly_summary/ (CSV for dashboards)")
print()
print("  How to reload for EDA:")
print("    df = spark.read.parquet('nyc_taxi_cleaned/all_months_cleaned/')")
print("    # or just one month:")
print("    df = spark.read.parquet('nyc_taxi_cleaned/all_months_cleaned/pickup_month=3/')")
print("=" * 55)
 
spark.stop()


· Exporting cleaned data (all months)
  Writing partitioned parquet...
  ✓ Parquet → nyc_taxi_cleaned/all_months_cleaned/
      Structure: pickup_month=1/, pickup_month=2/ …

  Writing monthly summary CSV...
  ✓ Monthly summary CSV → nyc_taxi_cleaned/monthly_summary/

Phase 2 complete! All 12 months are clean.

  Your output folder:
    nyc_taxi_cleaned/
    ├── all_months_cleaned/
    │   ├── pickup_month=1/
    │   ├── pickup_month=2/
    │   └── ... (one folder per month)
    └── monthly_summary/ (CSV for dashboards)

  How to reload for EDA:
    df = spark.read.parquet('nyc_taxi_cleaned/all_months_cleaned/')
    # or just one month:
    df = spark.read.parquet('nyc_taxi_cleaned/all_months_cleaned/pickup_month=3/')
